# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

In [ ]:
%load_ext autoreload
%autoreload 2

import json, os

from _campaign_lib import *

svc = await init_services()
campaign_rounds = []

In [ ]:
campaign_config = {
    "queries_per_eval": 8,               # queries per optimization evaluation step
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "pipeline_overrides": {              # Override specific pipeline params (omitted = backend default):
        # "profiling_temperature": 0.3,
    },
    "optimization": {
        "n_variants": 5,
        "creativity": 0.7,
        "improvement_threshold": 0.01,
        "patience": 3,                   # rounds without improvement before auto-stop
        "max_rounds": 10,
    },
    "eval_llm": {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        "temperature": 0,
        "max_tokens": 4000,
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                   "descriptions to standardized database terms using entity profiling "
                   "and candidate ranking.",
        "grid_budget": 35,            # exact budget (0=full grid)
        "eval_queries_per_point": 1,  # queries per grid point (0=use all eval_data)
        "shared_queries": False,      # False=different random queries per point
        "seed": 42,
        "top_k": 5,
        "use_defaults": True,        # use DEFAULT_GRID_AXES library
    },
    "smart_search": {
        "n_diagnostic": 6,
        "max_rounds": 3,
        "stop_threshold": 0.0,
    },
}

In [ ]:
# Pipeline workflow config (synced from backend GET /pipeline)
pipeline_config = load_pipeline_config(svc["exp_data"])
print(json.dumps(pipeline_config, indent=2))

pipeline_params = build_pipeline_params(
    pipeline_config, overrides=campaign_config.get("pipeline_overrides"),
)
campaign_config["pipeline_params"] = pipeline_params

In [ ]:
#@title 📡 Langfuse — cloud sync config
# Credentials: set LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY in .env
# Project name sets the Langfuse dataset name for this campaign's eval data.
LANGFUSE_PROJECT_NAME = "termnorm_ground_truth"
LANGFUSE_BACKFILL = True  # True = push all historical runs now
LANGFUSE_RESET = False     # True = clear push state first (re-push everything)

import api.services.obs.langfuse_push as _lfp
_lfp.DATASET_NAME = LANGFUSE_PROJECT_NAME

if LANGFUSE_BACKFILL:
    if LANGFUSE_RESET:
        from api.services.obs.langfuse_push import _state_path, _fresh_state, _save_state
        _save_state(svc["store"], svc["backend_id"], _fresh_state())
        print("Langfuse push state reset — will re-push all runs.")
    stats = push_langfuse(svc["store"], svc["backend_id"])

Langfuse push state reset — will re-push all runs.
  LANGFUSE PUSH (dataset-first)
Found 117 completed dataset runs for 'termnorm-local'
  Dataset 'termnorm_ground_truth': 40 items (40 created, 0 updated)
  baseline: 1 runs
  grid_search: 70 runs


In [ ]:
eval_data = load_eval_dataset(svc["store"], svc["backend_id"], svc["experiment_id"])
cov_df = analyze_candidate_coverage(eval_data)

show_entity_profiles(eval_data)

In [ ]:
baseline = load_baseline_prompt(svc["exp_data"])
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
print(f"Evaluation data: {len(eval_data)} queries")

In [ ]:
#@title Evaluate baseline prompt
campaign_rounds, baseline_results = await run_baseline_eval(
    baseline, eval_data, campaign_config, svc,
)

## 4.5 Smart Prompt Search

In [ ]:
#@title 4.5a — Build diagnostic set (with resume)
variant_library = load_variant_library()
ss = campaign_config.get("smart_search", {})

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

_result = await resume_or_build_diagnostic(
    campaign_config, baseline, baseline_results,
    llm_client, llm_model,
    svc["store"], svc["backend_id"], eval_data,
    improvement_areas=campaign_config.get("improvement_areas", ""),
)
plan_id, search_baseline, diagnostic, diag_summary, cached_profiles = _result

In [168]:
#@title 4.5a.1 — Historical data audit
prompt_index = build_historical_index(svc["store"], svc["backend_id"])

# Try to synthesize sensitivity from grid data
if not cached_profiles:
    synth = synthesize_sensitivity(
        svc["store"], svc["backend_id"], prompt_index, diagnostic,
    )
    if synth:
        scan_df, axis_profiles = synth
        cached_profiles = axis_profiles
        print("Sensitivity derived from grid data — scan may be skippable.")

2026-02-25 13:08:40 INFO     [api.services.search.coverage] build_prompt_result_index: 113 runs -> 110 unique prompts, 488 total query results


In [169]:
#@title 4.5a.1b — Data inventory
inventory = show_data_inventory(prompt_index, svc["store"], svc["backend_id"])

  DATA INVENTORY  (110 prompts, 488 query results)
  Baselines: 3 plan baseline(s) — 81 queries cached

  Axis                    Prompts  Queries  Distinct values
  persona                      20       20                3
  task_intent                  17       17                2
  thinking_style               22       22                3
  answer_format                15       15                1

  Pipeline parameters (from sensitivity scans):
    max_token_candidates     3 values scanned  sensitivity: 0.000  [skip]
    ranking_sample_size      3 values scanned  sensitivity: 0.000  [skip]
    ranking_temperature      4 values scanned  sensitivity: 0.000  [skip]

  Identified: 36/110 prompts (114/488 queries) via stored plans
  Unmatched:  74 prompts (374 queries)


In [170]:
#@title 4.5a.2 — Coverage advisor
# Knobs: adjust these and re-run to see different strategies
min_queries = 6          # min queries per variant to count as "usable"
axis_requirements = None  # None = require all values; or e.g. {"persona": 2}

coverage = show_scan_coverage(
    search_baseline, variant_library, diagnostic,
    prompt_index,
    pipeline_params=campaign_config.get("pipeline_params"),
    min_queries=min_queries,
    axis_requirements=axis_requirements,
)

  COVERAGE ADVISOR  (min_queries=6)
  Baseline: 6/6 queries cached ✓

  Prompt field axes:
    persona                4 values | 0/4 required  ✗  (0 usable, 4 uncovered)
    task_intent            3 values | 3/3 required  ✓  (3 usable)
    thinking_style         4 values | 4/4 required  ✓  (4 usable)
    answer_format          2 values | 0/2 required  ✗  (0 usable, 2 uncovered)
    problem_description    1 value  | 0/1 required  ✗  (0 usable, 1 uncovered)

  Pipeline params (always need backend):
    ranking_temperature    4 variants × 6 queries = 24 calls
    max_token_candidates   3 variants × 6 queries = 18 calls
    ranking_sample_size    3 variants × 6 queries = 18 calls

  Summary: 42 cached, 102 still needed (42 prompt-field + 60 pipeline-param)
  >> 2/5 prompt field axes covered. Run scan to fill gaps on: persona, answer_format, problem_description, ranking_temperature, max_token_candidates, ranking_sample_size. Tip: lower min_queries or reduce axis_requirements to accept spars

In [ ]:
#@title 4.5b — Sensitivity scan
if cached_profiles:
    print(f"[RESUME] Sensitivity scan already complete, "
          f"loaded {len(cached_profiles)} axis profiles")
    scan_df = None  # Not needed for adaptive search
    axis_profiles = cached_profiles
    display_axis_profiles(axis_profiles)
else:
    scan_df, axis_profiles = await sensitivity_scan(
        search_baseline, variant_library, diagnostic, svc.get("backend_client"),
        user_focus=campaign_config.get("improvement_areas", ""),
        store=svc["store"], backend_id=svc["backend_id"],
        pipeline_params=campaign_config.get("pipeline_params"),
        session_terms=svc.get("session_terms"),
        plan_id=plan_id,
        prompt_result_index=prompt_index,
    )

In [172]:
#@title 4.5c — Select best from scan & seed campaign (no backend needed)
best_ps, best_params = select_scan_winner_notebook(
    scan_df, axis_profiles, search_baseline, variant_library,
    pipeline_params=campaign_config.get("pipeline_params"),
    store=svc["store"], backend_id=svc["backend_id"], plan_id=plan_id,
)

if best_params:
    campaign_config["pipeline_params"] = best_params
    print(f"Updated pipeline_params: {best_params}")

campaign_rounds.append({
    "round": "search",
    "label": f"smart_search ({best_ps.changes_description or best_ps.id[:12]})",
    "prompt_state": best_ps,
    "accuracy": campaign_rounds[0]["accuracy"] if campaign_rounds else 0.0,
    "hits": campaign_rounds[0].get("hits", 0) if campaign_rounds else 0,
    "total": campaign_rounds[0].get("total", 0) if campaign_rounds else 0,
    "results": campaign_rounds[0].get("results", []) if campaign_rounds else [],
})
display_progress(campaign_rounds)

No scan data available. Run sensitivity scan (4.5b) first.
Updated pipeline_params: {'steps': ['entity_profiling', 'token_matching', 'llm_ranking']}

Round    Accuracy   Rolling Avg    Trend
  0        17.5%        17.5%  -
  search    17.5%        17.5%  +0.0%  <-- plateau


## 4.6 Grid Search — Landscape Exploration (Alternative)

<details>
<summary>Skip if you used Smart Search (4.5) above. Expand for brute-force grid sweep.</summary>

**What:** Systematic sweep of the prompt configuration space (Layer 1 fields) using a cartesian product of default axis variations. Maps the accuracy landscape before hill-climbing.

**When to use:** When you want exhaustive coverage of the grid, or when Smart Search results look unreliable and you want independent validation.

**What you get:** Ranked starting points, which dimensions matter most (marginal stats), interaction effects between fields (heatmaps), and LLM-analyzed insights.

**How to read results:**
- **Ranked table** — best combos at the top; use the winner as your campaign seed
- **Marginal stats** — which axis values have the highest mean accuracy across all combos
- **Pairwise heatmaps** — green = good interaction, red = bad; look for synergies and conflicts
- **LLM analysis** — automated pattern recognition across the grid results

</details>

In [173]:
#@title 4.6 — Grid Campaign Overview (existing plans)
merge_plans = False  # Set True to combine results from multiple plans
grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
merged_grid_df = grid_overview.get("merged_grid_df")

Grid plans (2):
  [done] gridplan_49738431a914  34 points  (space=96, axes=persona,task_intent,thinking_style,answer_format)
  [run..] gridplan_ab8b4dc4c8d8  34 points  (space=96, axes=persona,task_intent,thinking_style,answer_format)
Loaded 40 eval queries
Eval runs: 113 completed runs, 23 in-progress
  run_id                        name                 model                      temp  accuracy  queries
  baseline_816203b2             Baseline             meta-llama/llama-4-mav...  0     17.5%     40     
  grid_842a9010                 grid_combo_0         meta-llama/llama-4-mav...  0     0.0%      35     
  grid_6759eb45                 grid_combo_1         meta-llama/llama-4-mav...  0     0.0%      35     
  grid_aa8001f4                 grid_point_0                                    0.0   0.0%      1      
  grid_19f42064                 grid_point_1                                    0.0   0.0%      1      
  grid_2470cf35                 grid_point_2                            

In [ ]:
#@title 4.6a — Build or resume grid plan + load eval data
gs = campaign_config["grid_search"]
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

(
    grid_plan_id, grid_points, grid_state_lookup,
    grid_axes, layer1_fields, grid_baseline,
) = await resume_or_build_grid(
    campaign_config, baseline, llm_client, llm_model,
    svc["store"], svc["backend_id"],
    improvement_areas=campaign_config.get("improvement_areas", ""),
)

# Load full eval_data for later optimization rounds
eval_data = load_eval_dataset(
    svc["store"], svc["backend_id"], svc["experiment_id"],
)
if not eval_data:
    raise RuntimeError(
        "No evaluation data found. Generate data first "
        "(e.g. run termnorm_backend.ipynb or another data source)."
    )

print(f"Grid points: {len(grid_points)}")
print(f"Plan ID: {grid_plan_id}")

In [ ]:
#@title 4.6c — Run grid search
grid_df = await run_grid_search(
    grid_points, grid_state_lookup, eval_data,
    campaign_config["eval_llm"],
    plan_id=grid_plan_id,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_client=svc.get("backend_client"),
    session_terms=svc.get("session_terms"),
    pipeline_params=campaign_config.get("pipeline_params"),
    eval_queries_per_point=gs.get("eval_queries_per_point", 1),
    shared_queries=gs.get("shared_queries", False),
    grid_seed=gs.get("seed", 42),
)

In [176]:
#@title 4.6d — Display grid results
_display_df = merged_grid_df if merged_grid_df is not None else grid_df
display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))


GRID RESULTS — TOP 5


,persona,task_intent,thinking_style,answer_format,accuracy,hits,total,errors
0,1,2,3,1,1.0,1,1,0
1,0,0,3,1,1.0,1,1,0
2,0,0,1,0,0.0,0,1,0
3,0,0,0,0,0.0,0,1,0
4,0,0,2,0,0.0,0,1,0



MARGINAL STATS (mean accuracy per axis value)

  persona:
    [1] 20.0%  You are a domain expert with deep knowledge of this field.
    [0] 7.1%  (empty)
    [2] 0.0%  You are a precise, analytical system that evaluates candidat
    [3] 0.0%  You are a careful assistant that considers all options befor

  task_intent:
    [2] 11.1%  Rank candidates by how well they match the concept described
    [0] 5.9%  (empty)
    [1] 0.0%  Your task is to identify the single best match from the cand

  thinking_style:
    [3] 40.0%  First understand the core concept, then evaluate each candid
    [0] 0.0%  (empty)
    [1] 0.0%  Think step by step.
    [2] 0.0%  Focus on semantic meaning, not surface-level word overlap.

  answer_format:
    [1] 13.3%  Rank all candidates from most to least relevant.
    [0] 0.0%  (empty)

PAIRWISE INTERACTION HEATMAPS

  persona vs task_intent:


task_intent,0,1,2
persona,,,
0,14.3%,0.0%,0.0%
1,0.0%,0.0%,33.3%
2,0.0%,0.0%,0.0%
3,0.0%,0.0%,0.0%



  persona vs thinking_style:


...

thinking_style,0,1,2,3
task_intent,,,,
0,0.0%,0.0%,0.0%,100.0%
1,0.0%,0.0%,0.0%,0.0%
2,0.0%,0.0%,0.0%,100.0%



  task_intent vs answer_format:


answer_format,0,1
task_intent,,
0,0.0%,12.5%
1,0.0%,0.0%
2,0.0%,25.0%



  thinking_style vs answer_format:


answer_format,0,1
thinking_style,,
0,0.0%,0.0%
1,0.0%,0.0%
2,0.0%,0.0%
3,0.0%,100.0%


In [177]:
#@title 4.6e � LLM analysis of grid results
_analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
grid_analysis = await analyze_grid_results(
    _analysis_df, grid_axes, llm_client, model=llm_model,
)

LLM ANALYSIS PROMPT
You are an optimization advisor. Analyze the results of a grid search over prompt configuration fields.

GRID AXES: ['persona', 'task_intent', 'thinking_style', 'answer_format']
TOTAL GRID POINTS: 34

TOP 5 GRID POINTS:
[
  {
    "persona": 1,
    "task_intent": 2,
    "thinking_style": 3,
    "answer_format": 1,
    "prompt_state_id": "ba98f90c555e4c038b438a8cb9d0beba",
    "hits": 1,
    "total": 1,
    "accuracy": 1.0,
    "errors": 0
  },
  {
    "persona": 0,
    "task_intent": 0,
    "thinking_style": 3,
    "answer_format": 1,
    "prompt_state_id": "fd7f003eedef48c68c07b04f4042b7cf",
    "hits": 1,
    "total": 1,
    "accuracy": 1.0,
    "errors": 0
  },
  {
    "persona": 0,
    "task_intent": 0,
    "thinking_style": 1,
    "answer_format": 0,
    "prompt_state_id": "5b41263563ae4739b5ff7dc93327ec5c",
    "hits": 0,
    "total": 1,
    "accuracy": 0.0,
    "errors": 0
  },
  {
    "persona": 0,
    "task_intent": 0,
    "thinking_style": 0,
    "answer_fo

In [178]:
#@title 4.6f — Select grid winner and seed campaign
grid_winner = select_and_seed_grid_winner(
    grid_df, merged_grid_df, grid_state_lookup,
    grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
)

2026-02-25 13:08:44 WARNING  [api.services.search.grid_core] Winner based on only 1 queries — may be unreliable.



Layer 1 breakdown of grid winner:
  persona: You are a domain expert with deep knowledge of this field.
  task_intent: Rank candidates by how well they match the concept described.
  problem_description: Raw material descriptions are matched to standardized database terms using entit...
  instruction: Analyze the current entity profiling schema and suggest improvements to enhance ...
  thinking_style: First understand the core concept, then evaluate each candidate against it.
  answer_format: Rank all candidates from most to least relevant.

Rendered prompt preview (648 chars):
You are a domain expert with deep knowledge of this field.

Rank candidates by how well they match the concept described.

Raw material descriptions are matched to standardized database terms using entity profiling and candidate ranking, but profile schema quality and web search relevance need improvement

Analyze the current entity profiling schema and suggest improvements to enhance profile schema quality. Th

## 5. Run Optimization

<details>
<summary>Details</summary>

Two modes:
- **Semi-automatic** (recommended): runs multiple rounds with patience-based auto-stop
- **Manual**: run one round at a time for full HITL control

Both modes subsample `eval_data` to `queries_per_eval` queries per step.

</details>

In [179]:
#@title Run optimization (feedback cycle — M3 nodes)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
)

Feedback cycle: baseline acc=17.5%, max_rounds=10, patience=3


2026-02-25 13:08:44 INFO     [api.services.feedback_cycle] Using provided baseline (acc=0.175)
2026-02-25 13:08:44 INFO     [api.services.feedback_cycle] Feedback cycle round 0 (acc=0.175, stall=0/3)


  [1] C1/5 Q1/8 MISS PA66-GF25 Ultramid A3UG5 black 23215 RAL 9005
  [2] C1/5 Q2/8 MISS Round couper wire 2,5mm²/pre-welded braid
  [3] C1/5 Q3/8 MISS PEI-GF30 ULTEM 4000 black 7401/molding
  [4] C1/5 Q4/8 MISS Round couper wire 0,75mm²/pre-welded braid
  [5] C1/5 Q5/8 MISS Bimetal TB 20110/stamping
  [6] C1/5 Q6/8 MISS AgC5/stamping
  [7] C1/5 Q7/8 MISS PA66-Bergamid A700 CF RAL6018/molding
  [8] C1/5 Q8/8 MISS 6.8. - ISO 898-1/thread rolling
  >> Candidate 1/5 done: 0.0%
  [9] C2/5 Q1/8 HIT  PA66-GF25 Ultramid A3UG5 black 23215 RAL 9005
  [10] C2/5 Q2/8 HIT  Round couper wire 2,5mm²/pre-welded braid
  [11] C2/5 Q3/8 MISS PEI-GF30 ULTEM 4000 black 7401/molding
  [12] C2/5 Q4/8 MISS Round couper wire 0,75mm²/pre-welded braid
  [13] C2/5 Q5/8 MISS Bimetal TB 20110/stamping
  [14] C2/5 Q6/8 MISS AgC5/stamping
  [15] C2/5 Q7/8 MISS PA66-Bergamid A700 CF RAL6018/molding
  [16] C2/5 Q8/8 MISS 6.8. - ISO 898-1/thread rolling
  >> Candidate 2/5 done: 25.0%
  [17] C3/5 Q1/8 HIT  PA66-GF25 Ultr

CancelledError: 

In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, svc,
)

## 6. LLM Suggestion for Next Round (HITL)

<details>
<summary>Details</summary>

After each round, the LLM analyzes failures and suggests:
1. Failure pattern analysis
2. Parameter change suggestions
3. Prompt phrase fragments to adopt
4. Suggested next `campaign_config`

**Review the suggestions, edit the config cell (Section 2), then re-run Sections 5-6.**

</details>

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print(f"
--- SUGGESTED CONFIG (copy to Section 2) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

## 7. Campaign Summary

<details>
<summary>Details</summary>

Compare all rounds, track per-query flips, display the PromptState lineage chain, and save the winner.

</details>

In [ ]:
#@title Campaign comparison table
rows = []
for rd in campaign_rounds:
    rows.append({
        "round": rd["round"],
        "label": rd["label"][:40],
        "hit@1": rd["hits"],
        "total": rd["total"],
        "accuracy": f"{rd['accuracy']:.1%}",
        "prompt_id": rd["prompt_state"].id[:12],
    })

print(f"CAMPAIGN SUMMARY ({len(campaign_rounds)} rounds)")
print(f"{'='*70}")
display(pd.DataFrame(rows))

In [ ]:
#@title Per-query flip tracking (baseline vs final)
if len(campaign_rounds) >= 2:
    base_r = campaign_rounds[0]["results"]
    final_r = campaign_rounds[-1]["results"]

    flips = []
    for br, fr in zip(base_r, final_r):
        b_hit = br["hit"]
        f_hit = fr["hit"]
        if b_hit != f_hit:
            flips.append({
                "query": br["query"][:50],
                "flip": "MISS->HIT" if f_hit else "HIT->MISS",
                "base_pred": br["predicted"][:35],
                "final_pred": fr["predicted"][:35],
                "ground_truth": br["ground_truth"][:35],
            })

    gained = sum(1 for f in flips if f["flip"] == "MISS->HIT")
    lost = sum(1 for f in flips if f["flip"] == "HIT->MISS")

    print(f"FLIP TRACKING (baseline -> round {campaign_rounds[-1]['round']})")
    print(f"  Queries gained (MISS->HIT): {gained}")
    print(f"  Queries lost (HIT->MISS):   {lost}")
    print(f"  Net change:                 {gained - lost:+d}")
    print()
    if flips:
        display(pd.DataFrame(flips))
else:
    print("Need at least 2 rounds for flip tracking.")

In [ ]:
#@title PromptState lineage chain
print("LINEAGE CHAIN")
print("="*50)
for i, rd in enumerate(campaign_rounds):
    ps = rd["prompt_state"]
    parent = ps.parent_id[:12] if ps.parent_id else "root"
    arrow = "  " if i == 0 else "  -> "
    print(f"{arrow}[{ps.id[:12]}] Round {rd['round']}: {rd['label'][:40]} ({rd['accuracy']:.1%})")
    if ps.parent_id:
        print(f"       parent: {parent}  |  changes: {ps.changes_description or 'none'}")

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])

In [ ]:
#@title 📡 Sync evaluation history to Langfuse
# Backfill: pushes any eval runs not already synced to Langfuse cloud.
# Safe to re-run — already-pushed runs are skipped automatically.
import api.services.obs.langfuse_push as _lfp
_lfp.DATASET_NAME = LANGFUSE_PROJECT_NAME
stats = push_langfuse(svc["store"], svc["backend_id"])